# Predicting Laptop Prices with Feature Engineering
## Solution

**Short name (GitHub):** `LaptopPrice`

Worked answers for `LaptopPrice_Practice_Skeleton.ipynb`. Numbers below come from `random_state=42`, sklearn 1.5, default `RandomForestRegressor` (100 trees).

Hold-out (165 rows): **MAE ≈ ₹12,401**, **RMSE ≈ ₹23,165**, **R² ≈ 0.725**. Mean-predictor baseline MAE ≈ **₹31,232**. Train R² ≈ 0.966 — the forest memorizes, the gap is the usual bagged-tree pattern on a small card.


## Inline cheat-sheet

Same rules as the skeleton. Extra observed facts from this catalog:

| Column | After clean | Role |
|--------|-------------|------|
| `ram_gb` | 4 / 8 / 16 / 32 | strongest *raw numeric* corr with Price (0.52) |
| `graphic_card_gb` | 0 / 2 / 4 / 6 / 8 | corr 0.46; 0 = integrated |
| `total_storage` | SSD+HDD GB | corr 0.28 |
| `processor_name` | 11 families | target-encoded; dominates impurity importance (~0.56) |
| `brand` | 8 names | high-card target encode |
| `ram_type` | 6 types | high-card target encode |
| low-card one-hots | brand-of-CPU, OS, weight, warranty, touch, Office | `drop_first=True` |


## Flowchart

![flow](laptopprice_flowchart.png)


## 0. Packages


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

sns.set_theme(style="whitegrid")


## 1. Load and inspect


In [ ]:
df = pd.read_csv("data/laptop_price.csv")
print(df.head())
print(df.shape)
print(df.info())
print(df.nunique())
print("object cols:", df.select_dtypes(include="object").columns.tolist())
print(df["Price"].describe())


## 2. Memory and storage


In [ ]:
for col in ["ram_gb", "ssd", "hdd", "graphic_card_gb"]:
    df[col] = df[col].astype(str).str.replace(" GB", "", regex=False).astype(int)

df["total_storage"] = df["ssd"] + df["hdd"]
df = df.drop(columns=["ssd", "hdd"])
print(df[["ram_gb", "total_storage", "graphic_card_gb"]].head())
print(df[["ram_gb", "total_storage", "graphic_card_gb"]].describe())


## 3. OS bit, rating, generation


In [ ]:
df["os_bit"] = (
    df["os_bit"].astype(str).str.replace("-bit", "", regex=False).astype(int)
)
df["rating"] = (
    df["rating"].astype(str)
    .str.replace(" stars", "", regex=False)
    .str.replace(" star", "", regex=False)
    .astype(int)
)
df["processor_gnrtn"] = (
    df["processor_gnrtn"].astype(str)
    .str.replace("Not Available", "0", regex=False)
    .str.replace("th", "", regex=False)
    .astype(int)
)
print(df[["os_bit", "rating", "processor_gnrtn"]].head())
print("generation value counts:\n", df["processor_gnrtn"].value_counts())


## 4. EDA


In [ ]:
numeric_df = df.select_dtypes(include="number")
print("corr vs Price:\n", numeric_df.corr()["Price"].sort_values(ascending=False).round(3))

plt.figure(figsize=(11, 8))
sns.heatmap(numeric_df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation heatmap of numeric features")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(x="ram_gb", y="Price", data=df, order=sorted(df["ram_gb"].unique()))
plt.title("Price by RAM capacity")
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(x="graphic_card_gb", y="Price", data=df, order=sorted(df["graphic_card_gb"].unique()))
plt.title("Price by dedicated GPU memory")
plt.show()

plt.figure(figsize=(8, 5))
sns.histplot(df["Price"], bins=40, kde=True)
plt.axvline(df["Price"].median(), color="crimson", ls="--", label="median")
plt.axvline(df["Price"].mean(), color="navy", ls=":", label="mean")
plt.title("Price is right-skewed")
plt.legend()
plt.show()


Expected read: `ram_gb` 0.52 and `graphic_card_gb` 0.46 lead the numeric correlations. Review *counts* correlate **negatively** with price (cheap, high-volume SKUs collect more reviews). `os_bit` and `rating` are nearly flat. Median price steps up with RAM; the 0 GB GPU cluster sits well below 4/6/8 GB discrete cards.


## 5. Encode + split (lesson version, full-frame target means)


In [ ]:
work = df.copy()
cat_cols = work.select_dtypes(include=["object"]).columns.tolist()
print({c: work[c].nunique() for c in cat_cols})

for col in cat_cols:
    if work[col].nunique() < 5:
        dummies = pd.get_dummies(work[col], prefix=col, drop_first=True)
        work = pd.concat([work, dummies], axis=1)
        work.drop(columns=[col], inplace=True)
    else:
        work[col] = work[col].map(work.groupby(col)["Price"].mean())

X = work.drop(columns=["Price"])
y = work["Price"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("X", X.shape, "train", X_train.shape, "test", X_test.shape)
print(X.columns.tolist())


## 6. Random Forest


In [ ]:
rf_model = RandomForestRegressor(random_state=42)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)
print("train R²", round(r2_score(y_train, rf_model.predict(X_train)), 4))


## 7. Evaluate + top features


In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
base_mae = mean_absolute_error(y_test, np.full_like(y_test, y_train.mean(), dtype=float))
print(f"MAE: {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R²: {r2:.4f}")
print(f"mean-baseline MAE: {base_mae:,.2f}")

importances = rf_model.feature_importances_
feature_names = np.array(X_train.columns)
top_5_idx = np.argsort(importances)[-5:][::-1]
print(pd.Series(importances[top_5_idx], index=feature_names[top_5_idx]))

plt.figure(figsize=(8, 5))
sns.barplot(x=importances[top_5_idx], y=feature_names[top_5_idx], color="#1F4E79")
plt.title("Top 5 impurity importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, s=28, alpha=0.55)
lo, hi = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
plt.plot([lo, hi], [lo, hi], "r--")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title(f"Hold-out  ·  MAE={mae:,.0f}  R²={r2:.3f}")
plt.show()


### Observed numbers (rs=42, lesson encoding)

| Metric | Value |
|--------|-------|
| Test MAE | **₹12,585** (leaky encode) / **₹12,401** (train-only encode) |
| Test RMSE | ≈ ₹23.2k |
| Test R² | **0.724 – 0.725** |
| Train R² | ≈ 0.966 |
| Mean baseline MAE | **₹31,232** |
| Median baseline MAE | ₹29,881 |
| LinearRegression on same matrix | MAE ₹16,450 / R² 0.660 |
| Ridge(α=1) | MAE ₹16,538 / R² 0.657 |

Leakage barely moved the score here because every brand / CPU family appears on both sides of the split. That is catalog luck, not a reason to skip the safe pattern.

Top impurity: `processor_name` (~0.56), `total_storage` (~0.12), `graphic_card_gb`, `Number of Ratings`, `ram_type`. Permutation on the test fold still puts `processor_name` first, then storage, ratings-count, GPU, and **RAM** (RAM rises once the encoding bias is removed).


## 8. Alternate code


### 8a. Regex extract


In [ ]:
# Re-load a raw copy if you already stripped units
raw = pd.read_csv("data/laptop_price.csv")
for col in ["ram_gb", "ssd", "hdd", "graphic_card_gb"]:
    raw[col] = raw[col].astype(str).str.extract(r"(\d+)", expand=False).astype(int)
print(raw[["ram_gb", "ssd", "hdd", "graphic_card_gb"]].head())


### 8b. Leakage-safe encoding


In [ ]:
y_safe = df["Price"]
X_raw = df.drop(columns=["Price"])
Xtr, Xte, ytr, yte = train_test_split(X_raw, y_safe, test_size=0.2, random_state=42)

cat_cols = Xtr.select_dtypes(include=["object"]).columns.tolist()
low = [c for c in cat_cols if df[c].nunique() < 5]
high = [c for c in cat_cols if df[c].nunique() >= 5]
print("one-hot", low)
print("target-encode", high)

Xtr_enc, Xte_enc = Xtr.copy(), Xte.copy()
for col in high:
    means = pd.concat([Xtr_enc[col], ytr], axis=1).groupby(col)[ytr.name].mean()
    Xtr_enc[col] = Xtr_enc[col].map(means)
    Xte_enc[col] = Xte_enc[col].map(means).fillna(ytr.mean())
for col in low:
    dtr = pd.get_dummies(Xtr_enc[col], prefix=col, drop_first=True)
    dte = pd.get_dummies(Xte_enc[col], prefix=col, drop_first=True)
    dte = dte.reindex(columns=dtr.columns, fill_value=0)
    Xtr_enc = pd.concat([Xtr_enc.drop(columns=[col]), dtr], axis=1)
    Xte_enc = pd.concat([Xte_enc.drop(columns=[col]), dte], axis=1)

rf_safe = RandomForestRegressor(random_state=42)
rf_safe.fit(Xtr_enc, ytr)
yp_safe = rf_safe.predict(Xte_enc)
print("SAFE MAE", round(mean_absolute_error(yte, yp_safe), 2),
      "R²", round(r2_score(yte, yp_safe), 4))


### 8c. Linear and Ridge


In [ ]:
lr = LinearRegression().fit(Xtr_enc, ytr)
rg = Ridge(alpha=1.0).fit(Xtr_enc, ytr)
print("LR   MAE", round(mean_absolute_error(yte, lr.predict(Xte_enc)), 2),
      "R²", round(r2_score(yte, lr.predict(Xte_enc)), 4))
print("Ridge MAE", round(mean_absolute_error(yte, rg.predict(Xte_enc)), 2),
      "R²", round(r2_score(yte, rg.predict(Xte_enc)), 4))


### 8d. Permutation importance


In [ ]:
perm = permutation_importance(
    rf_safe, Xte_enc, yte, n_repeats=8, random_state=42
)
pimp = pd.Series(perm.importances_mean, index=Xte_enc.columns).sort_values(ascending=False)
print(pimp.head(8).round(4))
pimp.head(8).iloc[::-1].plot(kind="barh", figsize=(8, 5), color="#C45C26")
plt.title("Permutation importance on the hold-out fold")
plt.tight_layout()
plt.show()


## 9. More practice — worked sketches


In [ ]:
# 9.1 log-price target
rf_log = RandomForestRegressor(random_state=42)
rf_log.fit(Xtr_enc, np.log1p(ytr))
yp_log = np.expm1(rf_log.predict(Xte_enc))
print("log-target MAE", round(mean_absolute_error(yte, yp_log), 2),
      "R²", round(r2_score(yte, yp_log), 4))

# 9.3 drop popularity columns
drop_pop = [c for c in Xtr_enc.columns if "Number of" in c]
rf_np = RandomForestRegressor(random_state=42)
rf_np.fit(Xtr_enc.drop(columns=drop_pop), ytr)
yp_np = rf_np.predict(Xte_enc.drop(columns=drop_pop))
print("no-popularity MAE", round(mean_absolute_error(yte, yp_np), 2),
      "R²", round(r2_score(yte, yp_np), 4))

# 9.5 premium tail (Price >= 100000)
premium = yte >= 100000
print("premium share in test", float(premium.mean()))
print("premium MAE", round(mean_absolute_error(yte[premium], yp_safe[premium]), 2) if premium.any() else None)
print("mass-market MAE", round(mean_absolute_error(yte[~premium], yp_safe[~premium]), 2))


## 10. Simulation


In [ ]:
N_EST = 100
MAX_DEPTH = None
NOISE_SD = 0
SUBSAMPLE = 1.0
RANDOM_STATE = 42

rng = np.random.default_rng(RANDOM_STATE)
n = int(len(Xtr_enc) * SUBSAMPLE)
idx = rng.choice(len(Xtr_enc), size=max(n, 20), replace=False)
X_s = Xtr_enc.iloc[idx]
y_s = ytr.iloc[idx].astype(float) + rng.normal(0, NOISE_SD, size=len(idx))

sim = RandomForestRegressor(
    n_estimators=N_EST, max_depth=MAX_DEPTH, random_state=RANDOM_STATE
)
sim.fit(X_s, y_s)
yp_sim = sim.predict(Xte_enc)
print(f"knobs  n_est={N_EST}  depth={MAX_DEPTH}  noise={NOISE_SD}  sub={SUBSAMPLE}")
print("sim MAE", round(mean_absolute_error(yte, yp_sim), 2),
      "R²", round(r2_score(yte, yp_sim), 4))

# small sweep for the chart
rows = []
for n_est in [10, 25, 50, 100, 200]:
    m = RandomForestRegressor(n_estimators=n_est, random_state=42)
    m.fit(Xtr_enc, ytr)
    p = m.predict(Xte_enc)
    rows.append((n_est, mean_absolute_error(yte, p), r2_score(yte, p)))
sweep = pd.DataFrame(rows, columns=["n_estimators", "MAE", "R2"])
print(sweep)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(sweep["n_estimators"], sweep["MAE"], "o-")
ax[0].set_title("Test MAE vs n_estimators")
ax[1].plot(sweep["n_estimators"], sweep["R2"], "o-")
ax[1].set_title("Test R² vs n_estimators")
plt.tight_layout()
plt.show()


On this card the forest is already flat by ~25–50 trees (MAE 12.1k–12.6k). Extra trees do not buy much. Depth caps around 4 underfit; `None` is fine at n=658. Adding ₹15k of label noise or cutting the train set to 40% both lift MAE toward the mid-teens of thousands — the spec signal is real but not infinite.


## 11. Can / cannot (solution notes)

The ₹12.4k MAE is about **16% of mean price** and **19% of median price**. That is useful for a first-pass listing screen and terrible as a checkout quote on a ₹4 lakh workstation (the tail is where residuals explode — see the pred-vs-actual chart).

`processor_name` target encoding is doing a lot of work that a merchandiser would call "SKU family." RAM's raw correlation is high, but once CPU family is in the model the forest spends fewer splits on RAM — permutation importance puts RAM back near GPU, which matches the boxplots.

Review counts are available on a *scraped* catalog and would not be available for a yet-unlisted config. Treat them as optional context, not as a design lever.
